# ContextQA - Phase 5: Official SQuAD Evaluation & Error Analysis

In this notebook, we perform the rigorous official SQuAD v1.1 evaluation on the validation split (10,570 questions).

### Core Metrics Evaluated:
1. **Exact Match (EM)**: Binary percentage of predictions that match the ground truth character-for-character (after normalization).
2. **Token-Level F1**: Harmonic mean of Precision and Recall over words, rewarding partial overlaps.
3. **Inference Latency**: Milliseconds per question on the RTX 3060 GPU.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import load_json

# Load evaluation metrics
metrics = load_json("results/metrics.json")
print("=" * 40)
print("📊 OFFICIAL SQUAD v1.1 EVALUATION METRICS:")
for k, v in metrics.items():
    print(f"  - {k:<25}: {v}")
print("=" * 40)

## 1. Error Analysis: Breakdown by Prediction Categories

In [ ]:
error_df = pd.read_csv("results/error_analysis.csv")
print(f"Loaded {len(error_df):,} evaluation records.")

cat_summary = error_df["error_category"].value_counts()
print("\nPrediction Category Distribution:")
print(cat_summary)

fig, ax = plt.subplots(figsize=(8, 4))
cat_summary.plot(kind="bar", color=["#2ca02c", "#ff7f0e", "#d62728"], ax=ax)
ax.set_title("SQuAD v1.1 Predictions Categorization", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of Questions")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 2. Inspecting Qualitative Error Cases (Partial Matches vs Misses)

In [ ]:
print("=== Sample Partial Matches (Boundary Shifts) ===")
partial_matches = error_df[error_df["error_category"] == "Partial Match"].head(3)
for _, row in partial_matches.iterrows():
    print(f"Question: {row['question']}")
    print(f"  Ground Truth: '{row['first_ground_truth']}'")
    print(f"  Predicted:    '{row['prediction']}' (F1: {row['f1_score']}%)\n")

print("=== Sample Complete Misses ===")
misses = error_df[error_df["error_category"] == "Complete Miss"].head(3)
for _, row in misses.iterrows():
    print(f"Question: {row['question']}")
    print(f"  Ground Truth: '{row['first_ground_truth']}'")
    print(f"  Predicted:    '{row['prediction']}'\n")